In [0]:
display(dbutils.fs.ls("/Volumes/workspace/bronze/raw_files"))

In [0]:
raw_path = "/Volumes/workspace/bronze/raw_files/archive/"

display(dbutils.fs.ls(raw_path))

In [0]:
from pyspark.sql import functions as F
files = {
    "olist_customers_dataset.csv": "olist_customers",
    "olist_geolocation_dataset.csv": "olist_geolocation",
    "olist_order_items_dataset.csv": "olist_order_items",
    "olist_order_payments_dataset.csv": "olist_order_payments",
    "olist_order_reviews_dataset.csv": "olist_order_reviews",
    "olist_orders_dataset.csv": "olist_orders",
    "olist_products_dataset.csv": "olist_products",
    "olist_sellers_dataset.csv": "olist_sellers",
    "product_category_name_translation.csv": "product_category_name_translation"
}

# Percorre o dicionário: file_name = nome do CSV, table_name = nome da tabela Delta
for file_name, table_name in files.items():

    # ---------- LEITURA DO CSV ----------
    df = (
        spark.read
        # A primeira linha do CSV tem os nomes das colunas
        .option("header", True)
        # Permite valores com quebra de linha dentro das aspas
        # (corrige os comentários dos reviews que quebravam a tabela)
        .option("multiLine", True)
        # Aspas dentro do texto são escapadas com outra aspa ("")
        .option("escape", '"')
        # Linhas malformadas vão para a coluna _rescued_data em vez de sumirem
        .option("rescuedDataColumn", "_rescued_data")
        # SEM inferSchema: tudo é lido como texto.
        # Na bronze o dado fica como veio; o CEP mantém o zero à esquerda.
        .csv(raw_path + file_name)
        # Adiciona as colunas de metadados de rastreabilidade
        .select(
            "*",                                                       # todas as colunas originais
            F.col("_metadata.file_path").alias("_source_file"),       # arquivo de origem de cada linha
            F.current_timestamp().alias("_ingestion_timestamp")       # data/hora da carga
        )
    )

    # ---------- GRAVAÇÃO NA BRONZE ----------
    (
        df.write
        .format("delta")                      # formato Delta Lake
        .mode("overwrite")                    # substitui a tabela (rodar 2x não duplica dados)
        .option("overwriteSchema", True)      # permite trocar o schema antigo (INT -> texto)
        .saveAsTable(f"workspace.bronze.{table_name}")
    )

    # ---------- VALIDAÇÃO ----------
    # Conta as linhas da tabela já gravada (não relê o CSV)
    n = spark.table(f"workspace.bronze.{table_name}").count()
    print(f"{table_name} criada com {n} linhas")

In [0]:
%sql
-- 1. Reviews: o total deve ser ~99.224 e "invalidos" deve ser 0
SELECT count(*) AS total,
       count_if(review_score NOT IN ('1','2','3','4','5') OR review_score IS NULL) AS invalidos
FROM workspace.bronze.olist_order_reviews;


In [0]:
%sql
-- 2. CEP: o menor valor deve começar com zero (ex.: 01003)
SELECT min(customer_zip_code_prefix) FROM workspace.bronze.olist_customers;


In [0]:
%sql
-- 3. Linhas malformadas: o ideal é 0
SELECT count(*) FROM workspace.bronze.olist_order_reviews
WHERE _rescued_data IS NOT NULL;